# 📊 MODULE 3: FORECASTING AND MODEL SELECTION
## Time Series Analytics - Educational Notebook for Google Colab

---

### Topics Covered:
- **3.1** Train-Test Split for Time Series
- **3.2** AR Model Fitting and Forecasting
- **3.3** MA Model Fitting and Forecasting
- **3.4** ARMA Model Selection
- **3.5** ARIMA Model for Non-Stationary Data
- **3.6** Auto ARIMA
- **3.7** Residual Diagnostics
- **3.8** Rolling Forecast Validation
- **3.9** Prediction Intervals

---

**Instructions:** Run each cell in order. Each topic section is self-contained after the setup cell.

## 🔧 Setup: Install and Import Required Packages

Run this cell first to install dependencies and download data.

In [ ]:
# Install required packages (uncomment if running on Colab for the first time)
# !pip install yfinance pmdarima --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljung_box
from statsmodels.tsa.stattools import adfuller
from scipy import stats
import itertools

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All packages imported successfully!")

In [ ]:
# Download data
print("Downloading data...")
btc = yf.download('BTC-USD', start='2020-01-01', progress=False)['Close']
gold = yf.download('GC=F', start='2020-01-01', progress=False)['Close']
mrf = yf.download('MRF.NS', start='2020-01-01', progress=False)['Close']

# Clean data (remove NaN)
btc = btc.dropna()
gold = gold.dropna()
mrf = mrf.dropna()

print(f"✓ Bitcoin: {len(btc)} observations from {btc.index[0].date()} to {btc.index[-1].date()}")
print(f"✓ Gold: {len(gold)} observations from {gold.index[0].date()} to {gold.index[-1].date()}")
print(f"✓ MRF: {len(mrf)} observations from {mrf.index[0].date()} to {mrf.index[-1].date()}")
print("\n✅ Setup complete! Ready to run Module 3 topics.")

---

# 📌 TOPIC 3.1: Train-Test Split for Time Series

### Key Concepts:
- Time series data must be split **chronologically** (NOT randomly)
- Training data comes BEFORE test data
- Common splits: 80-20, 70-30, or based on specific date

⚠️ **NEVER shuffle time series data!** Time order contains critical information for forecasting.

In [ ]:
# Use Bitcoin data
data = btc.copy()

# Calculate split point (80-20 split chronologically)
split_idx = int(len(data) * 0.8)
train = data[:split_idx]
test = data[split_idx:]

# Print split information
print("Training Set:")
print(f"  - Size: {len(train)} observations")
print(f"  - Date Range: {train.index[0].date()} to {train.index[-1].date()}")
print(f"\nTest Set:")
print(f"  - Size: {len(test)} observations")
print(f"  - Date Range: {test.index[0].date()} to {test.index[-1].date()}")

In [ ]:
# Plot train and test sets
plt.figure(figsize=(12, 5))
plt.plot(train.index, train, label='Training Set (80%)', color='blue', linewidth=1.5)
plt.plot(test.index, test, label='Test Set (20%)', color='red', linewidth=1.5)
plt.axvline(x=train.index[-1], color='green', linestyle='--', linewidth=2, label='Split Point')
plt.title('Time Series Train-Test Split (Chronological)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Bitcoin Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- Training data: Used to fit the model
- Test data: Used to evaluate forecast accuracy
- The split must respect temporal order to avoid **data leakage**

---

# 📌 TOPIC 3.2: AR Model Fitting and Forecasting

### Autoregressive (AR) Model:
An AR(p) model predicts future values using **p past values**.

**Formula:** $Y_t = \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + ... + \phi_p Y_{t-p} + \epsilon_t$

- $\phi_i$ = AR coefficients
- $\epsilon_t$ = White noise error term

In [ ]:
# Use Gold returns (make stationary)
gold_returns = gold.pct_change().dropna() * 100  # Convert to percentage
train_returns = gold_returns[:int(len(gold_returns)*0.8)]
test_returns = gold_returns[int(len(gold_returns)*0.8):]

print(f"Training returns: {len(train_returns)} observations")
print(f"Test returns: {len(test_returns)} observations")

In [ ]:
# Fit AR(2) model: ARIMA(2,0,0)
ar_model = ARIMA(train_returns, order=(2, 0, 0))
ar_fitted = ar_model.fit()

# Print model summary
print("AR(2) Model Summary:")
print(f"  AR Coefficient 1 (φ₁): {ar_fitted.params[1]:.4f}")
print(f"  AR Coefficient 2 (φ₂): {ar_fitted.params[2]:.4f}")
print(f"  AIC: {ar_fitted.aic:.2f}")
print(f"  BIC: {ar_fitted.bic:.2f}")

In [ ]:
# Forecast next 30 days
forecast_steps = 30
forecast = ar_fitted.forecast(steps=forecast_steps)
forecast_ci = ar_fitted.get_forecast(steps=forecast_steps).conf_int()

# Create forecast index
last_date = train_returns.index[-1]
forecast_index = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_steps, freq='D')

# Plot actual vs forecast
plt.figure(figsize=(12, 5))
plt.plot(train_returns.index[-100:], train_returns[-100:], label='Training Data', color='blue')
plt.plot(test_returns.index[:30], test_returns[:30], label='Actual Test Data', color='green')
plt.plot(forecast_index, forecast, label='AR(2) Forecast', color='red', linewidth=2)
plt.fill_between(forecast_index, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1], 
                 color='red', alpha=0.2, label='95% Confidence Interval')
plt.title('AR(2) Model: Gold Returns Forecast', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Returns (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- AR(2) uses the **last 2 observations** to predict the next value
- AR models capture **momentum** or **mean reversion** in the data
- The forecast converges to the mean as the horizon increases

---

# 📌 TOPIC 3.3: MA Model Fitting and Forecasting

### Moving Average (MA) Model:
An MA(q) model predicts future values using **q past forecast errors (shocks)**.

**Formula:** $Y_t = \epsilon_t + \theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2} + ... + \theta_q \epsilon_{t-q}$

- $\theta_i$ = MA coefficients
- $\epsilon_t$ = White noise shock at time t

In [ ]:
# Use MRF returns
mrf_returns = mrf.pct_change().dropna() * 100
train_mrf = mrf_returns[:int(len(mrf_returns)*0.8)]

# Fit MA(2) model: ARIMA(0,0,2)
ma_model = ARIMA(train_mrf, order=(0, 0, 2))
ma_fitted = ma_model.fit()

# Print model parameters
print("MA(2) Model Parameters:")
print(f"  MA Coefficient 1 (θ₁): {ma_fitted.params[1]:.4f}")
print(f"  MA Coefficient 2 (θ₂): {ma_fitted.params[2]:.4f}")
print(f"  AIC: {ma_fitted.aic:.2f}")
print(f"  BIC: {ma_fitted.bic:.2f}")

In [ ]:
# Make 20-step ahead forecast
forecast_ma = ma_fitted.forecast(steps=20)
forecast_ma_ci = ma_fitted.get_forecast(steps=20).conf_int()
forecast_ma_index = pd.date_range(start=train_mrf.index[-1] + pd.Timedelta(days=1), periods=20, freq='D')

# Plot with confidence bands
plt.figure(figsize=(12, 5))
plt.plot(train_mrf.index[-100:], train_mrf[-100:], label='Training Data', color='blue')
plt.plot(forecast_ma_index, forecast_ma, label='MA(2) Forecast', color='red', linewidth=2, marker='o')
plt.fill_between(forecast_ma_index, forecast_ma_ci.iloc[:, 0], forecast_ma_ci.iloc[:, 1],
                 color='red', alpha=0.2, label='95% Confidence Bands')
plt.title('MA(2) Model: MRF Returns Forecast', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Returns (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- MA(2) models the effect of **past shocks/innovations**
- MA forecasts quickly converge to the mean (after q steps)
- Useful for capturing **short-term shock effects**

---

# 📌 TOPIC 3.4: ARMA Model Selection

### Choosing the Best (p, q):
We compare multiple ARMA(p,q) models using information criteria:

- **AIC (Akaike Information Criterion)**: Balances fit and complexity
- **BIC (Bayesian Information Criterion)**: Penalizes complexity more heavily

**Lower values = Better model**

In [ ]:
# Use Bitcoin returns
btc_returns = btc.pct_change().dropna() * 100
train_btc = btc_returns[:int(len(btc_returns)*0.8)]

# Fit multiple ARMA(p,q) models
results = []
print("Fitting ARMA models...")
for p in [0, 1, 2, 3]:
    for q in [0, 1, 2, 3]:
        if p == 0 and q == 0:
            continue  # Skip ARMA(0,0)
        try:
            model = ARIMA(train_btc, order=(p, 0, q))
            fitted = model.fit()
            results.append({
                'Model': f'ARMA({p},{q})',
                'p': p,
                'q': q,
                'AIC': fitted.aic,
                'BIC': fitted.bic
            })
        except:
            pass

print("Done!")

In [ ]:
# Create comparison table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('AIC')

print("ARMA Model Comparison (sorted by AIC):")
print("="*50)
print(results_df.to_string(index=False))

In [ ]:
# Find best models
best_aic = results_df.iloc[0]
best_bic = results_df.sort_values('BIC').iloc[0]

print(f"\n✓ Best model by AIC: {best_aic['Model']} (AIC = {best_aic['AIC']:.2f})")
print(f"✓ Best model by BIC: {best_bic['Model']} (BIC = {best_bic['BIC']:.2f})")

### 📝 Key Takeaway:
- **AIC**: Tends to select slightly more complex models
- **BIC**: Prefers simpler models (stronger penalty for parameters)
- When AIC and BIC disagree, consider the trade-off between fit and parsimony

---

# 📌 TOPIC 3.5: ARIMA Model for Non-Stationary Data

### ARIMA(p, d, q):
The **d** parameter represents the order of differencing needed to make the series stationary.

- **d = 0**: Series is already stationary (use ARMA)
- **d = 1**: First difference makes it stationary
- **d = 2**: Second difference needed (rare)

We use the **ADF test** to determine if differencing is needed.

In [ ]:
# Use raw MRF stock price (non-stationary)
train_mrf_price = mrf[:int(len(mrf)*0.8)]
test_mrf_price = mrf[int(len(mrf)*0.8):]

# Determine d using ADF test
adf_result = adfuller(train_mrf_price)
print("ADF Test on MRF Price:")
print(f"  Test Statistic: {adf_result[0]:.4f}")
print(f"  p-value: {adf_result[1]:.4f}")
print(f"  Critical Values:")
for key, value in adf_result[4].items():
    print(f"    {key}: {value:.4f}")

if adf_result[1] > 0.05:
    print("\n→ Series is NON-STATIONARY (need differencing, d=1)")
    d = 1
else:
    print("\n→ Series is STATIONARY (d=0)")
    d = 0

In [ ]:
# Fit ARIMA(1,1,1) model
arima_model = ARIMA(train_mrf_price, order=(1, d, 1))
arima_fitted = arima_model.fit()

print(f"ARIMA(1,{d},1) Model fitted.")
print(f"  AIC: {arima_fitted.aic:.2f}")
print(f"  BIC: {arima_fitted.bic:.2f}")

In [ ]:
# Forecast next 30 days
forecast_arima = arima_fitted.forecast(steps=30)
forecast_arima_ci = arima_fitted.get_forecast(steps=30).conf_int()
forecast_arima_index = pd.date_range(start=train_mrf_price.index[-1] + pd.Timedelta(days=1), 
                                     periods=30, freq='D')

# Plot actual vs forecast
plt.figure(figsize=(12, 5))
plt.plot(train_mrf_price.index[-100:], train_mrf_price[-100:], label='Training Data', color='blue')
plt.plot(test_mrf_price.index[:30], test_mrf_price[:30], label='Actual Test', color='green')
plt.plot(forecast_arima_index, forecast_arima, label='ARIMA Forecast', color='red', linewidth=2)
plt.fill_between(forecast_arima_index, forecast_arima_ci.iloc[:, 0], forecast_arima_ci.iloc[:, 1],
                 color='red', alpha=0.2)
plt.title('ARIMA(1,1,1) Model: MRF Price Forecast', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (INR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- ARIMA can handle **non-stationary** data through differencing
- The **I** in ARIMA stands for "Integrated" (inverse of differencing)
- Most financial price series need d=1 (first differencing)

---

# 📌 TOPIC 3.6: Auto ARIMA

### Automatic Model Selection:
The `auto_arima` function from `pmdarima` automatically:
1. Tests for stationarity
2. Determines optimal d
3. Searches over p and q values
4. Returns the best model based on AIC/BIC

In [ ]:
# Install pmdarima if needed
try:
    from pmdarima import auto_arima
    print("✓ pmdarima package available")
except:
    print("Installing pmdarima...")
    !pip install pmdarima --quiet
    from pmdarima import auto_arima
    print("✓ pmdarima installed")

In [ ]:
# Use Gold price data
train_gold = gold[:int(len(gold)*0.8)]

# Run auto_arima to find best (p,d,q)
print("Running Auto ARIMA (this may take a moment)...")
auto_model = auto_arima(train_gold, 
                        seasonal=False,
                        stepwise=True,
                        suppress_warnings=True,
                        error_action='ignore',
                        max_p=3, max_q=3, max_d=2,
                        trace=False)

print(f"\n✓ Best model selected: ARIMA{auto_model.order}")
print(f"  AIC: {auto_model.aic():.2f}")
print(f"  BIC: {auto_model.bic():.2f}")

In [ ]:
# Model summary
print(auto_model.summary())

In [ ]:
# Forecast
forecast_auto = auto_model.predict(n_periods=30)
forecast_auto_index = pd.date_range(start=train_gold.index[-1] + pd.Timedelta(days=1), 
                                    periods=30, freq='D')

# Plot
plt.figure(figsize=(12, 5))
plt.plot(train_gold.index[-100:], train_gold[-100:], label='Training Data', color='blue')
plt.plot(forecast_auto_index, forecast_auto, label=f'Auto ARIMA{auto_model.order} Forecast', 
         color='red', linewidth=2, marker='o', markersize=3)
plt.title('Auto ARIMA: Gold Price Forecast', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Gold Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- `auto_arima` automates the model selection process
- Great for quick prototyping and baseline models
- Always validate results with domain knowledge

---

# 📌 TOPIC 3.7: Residual Diagnostics

### Checking Model Adequacy:
A good model should have residuals that behave like **white noise**:
- No autocorrelation
- Approximately normal distribution
- Constant variance

### Diagnostic Tools:
1. **Residual plot**: Check for patterns
2. **ACF of residuals**: Should show no significant lags
3. **Histogram**: Should be approximately normal
4. **Q-Q plot**: Should follow diagonal line
5. **Ljung-Box test**: Statistical test for autocorrelation

In [ ]:
# Fit ARIMA model to Bitcoin data
btc_train = btc[:int(len(btc)*0.8)]
model_diag = ARIMA(btc_train, order=(1, 1, 1))
fitted_diag = model_diag.fit()

# Extract residuals
residuals = fitted_diag.resid
print(f"Number of residuals: {len(residuals)}")
print(f"Mean of residuals: {residuals.mean():.4f}")
print(f"Std of residuals: {residuals.std():.4f}")

In [ ]:
# Create 2x2 diagnostic plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals over time
axes[0, 0].plot(residuals)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_title('Residuals Over Time', fontweight='bold')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].grid(True, alpha=0.3)

# 2. ACF of residuals
plot_acf(residuals, lags=30, ax=axes[0, 1])
axes[0, 1].set_title('ACF of Residuals', fontweight='bold')

# 3. Histogram of residuals
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7, density=True)
# Overlay normal distribution
x = np.linspace(residuals.min(), residuals.max(), 100)
axes[1, 0].plot(x, stats.norm.pdf(x, residuals.mean(), residuals.std()), 'r-', lw=2, label='Normal')
axes[1, 0].set_title('Histogram of Residuals', fontweight='bold')
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Perform Ljung-Box test on residuals
lb_test = acorr_ljung_box(residuals, lags=[10, 20, 30], return_df=True)
print("Ljung-Box Test Results:")
print("="*50)
print(lb_test)
print("\nInterpretation:")
print("  p-value > 0.05: No significant autocorrelation (good!)")
print("  p-value < 0.05: Significant autocorrelation present (model inadequate)")

### 📝 Key Takeaway:
- Good residuals = **white noise** (random, no pattern)
- If residuals show patterns, the model is missing something
- The Ljung-Box test provides a statistical measure of residual autocorrelation

---

# 📌 TOPIC 3.8: Rolling Forecast Validation

### Walk-Forward Validation:
A more realistic evaluation method that simulates real-world forecasting:

1. Train on window of size `w`
2. Forecast 1 step ahead
3. Move window forward by 1
4. Repeat

This avoids look-ahead bias and tests the model on multiple forecast origins.

In [ ]:
# Use last 100 days of Gold data
gold_last100 = gold[-100:].copy()

# Rolling forecast with 30-day window
window_size = 30
predictions = []
actuals = []

print(f"Running rolling forecast with {window_size}-day window...")
for i in range(window_size, len(gold_last100)):
    train_window = gold_last100[i-window_size:i]
    
    # Fit model on window
    model_roll = ARIMA(train_window, order=(1, 1, 1))
    fitted_roll = model_roll.fit()
    
    # Forecast 1-day ahead
    forecast_1day = fitted_roll.forecast(steps=1)[0]
    predictions.append(forecast_1day)
    actuals.append(gold_last100.iloc[i])

print(f"Generated {len(predictions)} rolling forecasts.")

In [ ]:
# Calculate metrics
predictions = np.array(predictions)
actuals = np.array(actuals)
rmse = np.sqrt(np.mean((predictions - actuals)**2))
mae = np.mean(np.abs(predictions - actuals))
mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

print(f"\nRolling Forecast Performance:")
print(f"  RMSE: ${rmse:.2f}")
print(f"  MAE: ${mae:.2f}")
print(f"  MAPE: {mape:.2f}%")

In [ ]:
# Plot rolling forecasts
forecast_dates = gold_last100.index[window_size:]
plt.figure(figsize=(12, 5))
plt.plot(forecast_dates, actuals, label='Actual', color='blue', linewidth=2)
plt.plot(forecast_dates, predictions, label='Rolling Forecast', color='red', 
         linewidth=2, alpha=0.7, linestyle='--')
plt.title('Rolling Forecast Validation (30-day window)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Gold Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- Rolling validation **simulates real-world forecasting**
- More reliable than single train-test split
- Shows how model performs across different market conditions

---

# 📌 TOPIC 3.9: Prediction Intervals

### Quantifying Forecast Uncertainty:
Point forecasts alone are not enough. We need to communicate **uncertainty**.

- **80% Confidence Interval**: We're 80% confident the true value falls within this range
- **95% Confidence Interval**: We're 95% confident (wider range)

**Key insight**: Uncertainty **grows** as forecast horizon increases!

In [ ]:
# Fit ARIMA to MRF data
mrf_train = mrf[:int(len(mrf)*0.85)]
model_pi = ARIMA(mrf_train, order=(1, 1, 1))
fitted_pi = model_pi.fit()

# Forecast 60 days ahead with confidence intervals
forecast_obj = fitted_pi.get_forecast(steps=60)
forecast_pi = forecast_obj.predicted_mean
ci_80 = forecast_obj.conf_int(alpha=0.2)  # 80% CI
ci_95 = forecast_obj.conf_int(alpha=0.05)  # 95% CI

forecast_pi_index = pd.date_range(start=mrf_train.index[-1] + pd.Timedelta(days=1), 
                                  periods=60, freq='D')

print("Forecast generated for 60 days ahead with 80% and 95% confidence intervals.")

In [ ]:
# Plot forecast with both interval bands
plt.figure(figsize=(12, 6))
plt.plot(mrf_train.index[-100:], mrf_train[-100:], label='Training Data', color='blue', linewidth=2)
plt.plot(forecast_pi_index, forecast_pi, label='Forecast', color='red', linewidth=2)
plt.fill_between(forecast_pi_index, ci_95.iloc[:, 0], ci_95.iloc[:, 1],
                 color='red', alpha=0.15, label='95% Confidence Interval')
plt.fill_between(forecast_pi_index, ci_80.iloc[:, 0], ci_80.iloc[:, 1],
                 color='red', alpha=0.3, label='80% Confidence Interval')
plt.title('Prediction Intervals: Uncertainty Grows with Horizon', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('MRF Price (INR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Show interval widths at different horizons
widths_80 = ci_80.iloc[:, 1] - ci_80.iloc[:, 0]
widths_95 = ci_95.iloc[:, 1] - ci_95.iloc[:, 0]

print("Prediction Interval Widths:")
print("="*50)
print(f"  1-day ahead:")
print(f"    80% CI width: ₹{widths_80.iloc[0]:,.2f}")
print(f"    95% CI width: ₹{widths_95.iloc[0]:,.2f}")
print(f"\n  30-day ahead:")
print(f"    80% CI width: ₹{widths_80.iloc[29]:,.2f}")
print(f"    95% CI width: ₹{widths_95.iloc[29]:,.2f}")
print(f"\n  60-day ahead:")
print(f"    80% CI width: ₹{widths_80.iloc[59]:,.2f}")
print(f"    95% CI width: ₹{widths_95.iloc[59]:,.2f}")

In [ ]:
# Visualize how uncertainty grows
plt.figure(figsize=(10, 5))
horizons = range(1, 61)
plt.plot(horizons, widths_80, label='80% CI Width', color='blue', linewidth=2)
plt.plot(horizons, widths_95, label='95% CI Width', color='red', linewidth=2)
plt.title('Prediction Interval Width vs. Forecast Horizon', fontsize=14, fontweight='bold')
plt.xlabel('Forecast Horizon (days)')
plt.ylabel('Interval Width (INR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- **Uncertainty grows** as forecast horizon increases
- Always report prediction intervals, not just point forecasts
- 95% CI is wider than 80% CI (higher confidence = wider interval)
- This is why **long-term forecasts are inherently uncertain**

---

# ✅ MODULE 3 COMPLETE!

## Summary of Key Concepts:

| Topic | Key Takeaway |
|-------|-------------|
| 3.1 Train-Test Split | Never shuffle time series; split chronologically |
| 3.2 AR Models | Use past values to predict; captures momentum |
| 3.3 MA Models | Use past shocks to predict; captures short-term effects |
| 3.4 ARMA Selection | Use AIC/BIC to compare models; lower is better |
| 3.5 ARIMA | Handles non-stationary data via differencing |
| 3.6 Auto ARIMA | Automates parameter selection |
| 3.7 Residual Diagnostics | Good model = white noise residuals |
| 3.8 Rolling Validation | More realistic forecast evaluation |
| 3.9 Prediction Intervals | Uncertainty grows with horizon |

---

**Next:** Module 4 - Advanced Models (GARCH, Multivariate Models)